In [150]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd
import statsmodels.formula.api as smf

In [270]:
def plot_scatter_3d(fig, X, y):
    fig.add_trace(go.Scatter3d(
        x=X[:, 0],
        y=X[:, 1],
        z=y,
        mode='markers',
        marker=dict(
            size=3,
            colorscale='Spectral',
            opacity=0.8,
            color=y,
        )
    ))

    return fig

def plot_regression_plane(fig, X1_mesh, X2_mesh, Y_mesh):
    z_min = float(Y_mesh.min())
    z_max = float(Y_mesh.max())

    cmin = min(0.0, z_min)
    cmax = max(1.0, z_max)
    
    range_span = cmax - cmin if cmax > cmin else 1.0
    norm_0 = (0.0 - cmin) / range_span
    norm_1 = (1.0 - cmin) / range_span
    
    colorscale = []

    # Red zone for Z < 0
    if norm_0 > 0:
        colorscale.extend([[0.0, 'red'], [max(0.0, norm_0), 'red']])

    # Light blue zone for 0 <= Z <= 1
    colorscale.extend([
        [np.clip(norm_0, 0.0, 1.0), 'lightblue'],
        [np.clip(norm_1, 0.0, 1.0), 'lightblue']
    ])

    # Red zone for Z > 1
    if norm_1 < 1.0:
        colorscale.extend([[min(1.0, norm_1), 'red'], [1.0, 'red']])

    fig.add_trace(go.Surface(
        x=X1_mesh,
        y=X2_mesh,
        z=Y_mesh,
        cmin=cmin,
        cmax=cmax,
        colorscale=colorscale,
        opacity=0.6,
        showscale=False,
        name='Regression Plane'
    ))

    return fig

def set_plot_layout(fig):
    fig.update_layout(
        margin=dict(l=0, r=0, b=20, t=0),
        height=600,
        scene=dict(
            xaxis_title='X₁',
            yaxis_title='X₂',
            zaxis_title='Y',
            xaxis=dict(showticklabels=False),
            yaxis=dict(showticklabels=False),
            zaxis=dict(
                tickmode='array',
                tickvals=[0, 1],
            ),
            camera=dict(eye=dict(x=-1.2, y=1.8, z=0.8)),
            aspectmode='manual',
            aspectratio=dict(x=1.1, y=1.1, z=1)
        ),
    )

    return fig

In [272]:
np.random.seed(235)

X = np.random.rand(100, 2)
y = 3 * X[:, 0] + 2 * X[:, 1] + np.random.normal(0, 0.8, 100)
y = np.array(y < 2).astype(int)

fig = go.Figure()
fig = plot_scatter_3d(fig, X, y)
fig = set_plot_layout(fig)
fig.show()

In [273]:
df = pd.DataFrame({'X_1': X[:, 0], 'X_2': X[:, 1], 'Y': y})

lin_reg = smf.ols(formula='Y ~ X_1 + X_2', data=df).fit()
print('Linear model params:\n')
print(lin_reg.params)

X1_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 20)
X2_grid = np.linspace(X[:, 1].min(), X[:, 1].max(), 20)
X1_mesh, X2_mesh = np.meshgrid(X1_grid, X2_grid)
Y_mesh = lin_reg.params['Intercept'] + lin_reg.params['X_1'] * X1_mesh + lin_reg.params['X_2'] * X2_mesh

fig = go.Figure()
fig = plot_scatter_3d(fig, X, y)
fig = plot_regression_plane(fig, X1_mesh, X2_mesh, Y_mesh)
fig = set_plot_layout(fig)
fig.show()

Linear model params:

Intercept    1.287357
X_1         -1.088319
X_2         -0.694167
dtype: float64


In [274]:
log_reg = smf.logit(formula='Y ~ X_1 + X_2', data=df).fit()
print('Logistic model params:\n')
print(log_reg.params)

X1_grid = np.linspace(X[:, 0].min(), X[:, 0].max(), 20)
X2_grid = np.linspace(X[:, 1].min(), X[:, 1].max(), 20)
X1_mesh, X2_mesh = np.meshgrid(X1_grid, X2_grid)
Y_lin = log_reg.params['Intercept'] + log_reg.params['X_1'] * X1_mesh + log_reg.params['X_2'] * X2_mesh
Y_mesh = 1 / (1 + np.exp(-Y_lin))

fig = go.Figure()
fig = plot_scatter_3d(fig, X, y)
fig = plot_regression_plane(fig, X1_mesh, X2_mesh, Y_mesh)
fig = set_plot_layout(fig)
fig.show()

Optimization terminated successfully.
         Current function value: 0.351891
         Iterations 7
Logistic model params:

Intercept    6.341538
X_1         -8.642866
X_2         -5.736065
dtype: float64
